# Charger le datraframe #

In [11]:
import pandas as pd
import unicodedata
df = pd.read_csv(
    "ncr_ride_bookings.csv",
    sep=",",
    quotechar='"',
    encoding="utf-8"
)


# Supprime les espaces au début et à la fin des noms de colonnes #

In [12]:
df.columns = df.columns.str.strip()

# Supprimer les doublons #

In [13]:
df = df.drop_duplicates()

# Supprimer les espaces en début de chaque valeur #

In [14]:
# Sélection des colonnes de type chaîne
string_cols = df.select_dtypes(include="object").columns

# Suppression des espaces en début et fin de chaque valeur
df[string_cols] = df[string_cols].apply(lambda col: col.str.strip())
# remplacer les espaces multiples par un seul
df[string_cols] = df[string_cols].apply(
    lambda col: col.str.replace(r"\s+", " ", regex=True).str.strip()
)
# vérifier que les espaces ont bien été supprimés
print(df.columns.tolist())

for col in string_cols:
    print(df[col].head())

C:\Users\yannb\AppData\Local\Temp\ipykernel_15084\3566595447.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df.select_dtypes(include="object").columns


['Date', 'Time', 'Booking ID', 'Booking Status', 'Customer ID', 'Vehicle Type', 'Pickup Location', 'Drop Location', 'Avg VTAT', 'Avg CTAT', 'Cancelled Rides by Customer', 'Reason for cancelling by Customer', 'Cancelled Rides by Driver', 'Driver Cancellation Reason', 'Incomplete Rides', 'Incomplete Rides Reason', 'Booking Value', 'Ride Distance', 'Driver Ratings', 'Customer Rating', 'Payment Method']
0    2024-03-23
1    2024-11-29
2    2024-08-23
3    2024-10-21
4    2024-09-16
Name: Date, dtype: str
0    12:29:38
1    18:01:39
2    08:56:10
3    17:17:25
4    22:08:00
Name: Time, dtype: str
0    "CNR5884300"
1    "CNR1326809"
2    "CNR8494506"
3    "CNR8906825"
4    "CNR1950162"
Name: Booking ID, dtype: str
0    No Driver Found
1         Incomplete
2          Completed
3          Completed
4          Completed
Name: Booking Status, dtype: str
0    "CID1982111"
1    "CID4604802"
2    "CID9202816"
3    "CID2610914"
4    "CID9933542"
Name: Customer ID, dtype: str
0            eBike
1    

# Uniformiser les booking id #

In [15]:
id_cols = ["Booking ID", "Customer ID"]

for col in id_cols:
    df[col] = df[col].str.replace('"', '', regex=False).str.strip()

# Uniformiser les dates #

In [16]:
import re
from datetime import datetime

def uniformiser_date(date):

    if not isinstance(date, str):
        return None

    date = date.strip()

    # Uniformiser les séparateurs
    date = re.sub(r"\s*-\s*", "-", date)
    date = re.sub(r"\s+", "-", date)
    date = date.replace("/", "-")

    formats = [
        "%Y-%m-%d",   # 1995-02-10
        "%m-%d-%Y",   # 09-21-1972
        "%d-%m-%Y",   # 23-07-2008
        "%d-%b-%y",   # 22-Feb-04
        "%d-%B-%Y",   # 22-February-2004
        "%d-%b-%Y",   # 22-Feb-2004
    ]

    for fmt in formats:
        try:
            d = datetime.strptime(date, fmt)
            return d.strftime("%Y/%m/%d")
        except ValueError:
            pass

    return None

df["Date"] = df["Date"].apply(uniformiser_date)

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  str    
 1   Time                               150000 non-null  str    
 2   Booking ID                         150000 non-null  str    
 3   Booking Status                     150000 non-null  str    
 4   Customer ID                        150000 non-null  str    
 5   Vehicle Type                       150000 non-null  str    
 6   Pickup Location                    150000 non-null  str    
 7   Drop Location                      150000 non-null  str    
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by Customer  10500 non-null 

In [ ]:
df.describe()